# 🧠 A.X.I.S — Autonomous eXecution & Intelligence System

**Run each cell from top to bottom.**  
After Cell 5, you will get a public URL — open it in any browser to start chatting.

| Feature | Details |
|---|---|
| LLM | Ollama (llama3:8b) on Colab T4 GPU |
| Memory | ChromaDB persistent vector store |
| Backend | FastAPI + Socket.io |
| Tunnel | ngrok public HTTPS URL |

In [ ]:
# CELL 1 — Install Ollama and start the server
import subprocess, os, time, requests

print('Installing Ollama...')
os.system('curl -fsSL https://ollama.com/install.sh | sh')

# The installer puts Ollama at /usr/local/bin/ollama.
# Python subprocess does NOT automatically see the updated PATH,
# so we must use the full binary path explicitly.
OLLAMA_BIN = '/usr/local/bin/ollama'
os.environ['PATH'] = f"/usr/local/bin:{os.environ.get('PATH', '')}"

if not os.path.exists(OLLAMA_BIN):
    raise RuntimeError(f'Ollama binary not found at {OLLAMA_BIN}. The install script may have failed.')

print(f'Ollama binary found: {OLLAMA_BIN}')
print('Starting Ollama server in background...')
subprocess.Popen([OLLAMA_BIN, 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)  # Wait for the server to bind to port 11434

try:
    r = requests.get('http://localhost:11434')
    print('Ollama running:', r.text.strip())
except Exception as e:
    print('Ollama not responding:', e)

In [ ]:
# CELL 2 — Pull the LLM model
# Takes ~5 min on first run (~4.7 GB). Cached on subsequent runs.
MODEL = 'llama3:8b'   # Change to 'mistral' or 'gemma:2b' for lighter models

print(f'Pulling {MODEL}...')
result = subprocess.run([OLLAMA_BIN, 'pull', MODEL], capture_output=True, text=True)
print(result.stdout[-500:] if result.stdout else 'Done')

models = subprocess.run([OLLAMA_BIN, 'list'], capture_output=True, text=True)
print('Installed models:')
print(models.stdout)

In [ ]:
# CELL 3 — Install Python packages
print('Installing Python dependencies...')
os.system('pip install -q fastapi uvicorn python-socketio chromadb pyngrok requests')
print('All packages installed.')

In [ ]:
# CELL 4 — Clone the AXIS_AI project from GitHub
REPO_URL = 'https://github.com/aaweshdas/AXIS-AI.git'

if not os.path.exists('/content/AXIS_AI'):
    print('Cloning A.X.I.S project...')
    os.system(f'git clone {REPO_URL} /content/AXIS_AI')
else:
    print('Project already cloned. Pulling latest...')
    os.system('git -C /content/AXIS_AI pull')

os.chdir('/content/AXIS_AI')
print('Working directory:', os.getcwd())
print('Files:', os.listdir('.'))

In [ ]:
# CELL 5 — Start FastAPI + Socket.io server and create ngrok tunnel
import threading, uvicorn
from pyngrok import ngrok
import sys

sys.path.insert(0, '/content/AXIS_AI')
from server import socket_app, MODEL

PORT = 8000

def run_server():
    uvicorn.run(socket_app, host='0.0.0.0', port=PORT, log_level='warning')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)

public_url = ngrok.connect(PORT).public_url
print('\n' + '='*55)
print(f'  A.X.I.S is LIVE at:')
print(f'  {public_url}/ui')
print('='*55)
print(f'  Model: {MODEL}')
print(f'  API Health: {public_url}/api/health')
print('='*55 + '\n')

# Patch the frontend HTML to use the public ngrok URL
html_path = '/content/AXIS_AI/static/index.html'
with open(html_path, 'r') as f:
    html = f.read()

patch = f'\n  <script>window.AXIS_SERVER_URL = "{public_url}";</script>\n'
if 'AXIS_SERVER_URL' not in html:
    html = html.replace('<script src="https://cdn.socket.io', patch + '  <script src="https://cdn.socket.io')
    with open(html_path, 'w') as f:
        f.write(html)
print('Frontend patched with public server URL.')

In [ ]:
# CELL 6 — Health check
import requests as req

print('Running health checks...\n')

try:
    r = req.get('http://localhost:11434', timeout=3)
    print('Ollama:', r.text.strip())
except:
    print('Ollama: NOT reachable')

try:
    r = req.get(f'http://localhost:{PORT}/api/health', timeout=5)
    d = r.json()
    print(f'FastAPI: online | model={d["model"]} | memories={d["memory_entries"]}')
except Exception as e:
    print(f'FastAPI: {e}')

try:
    tunnels = ngrok.get_tunnels()
    for t in tunnels:
        print(f'ngrok: {t.public_url} -> port {PORT}')
except:
    print('ngrok: no active tunnels')

print('\nOpen the URL from Cell 5 and start chatting!')